In [22]:
import pandas as pd
from pycaret.classification import *
from pycaret.regression import *


In [23]:
# Charger les fichiers CSV
df_large = pd.read_csv('../data/raw/bank-full.csv', sep=';')
df_small = pd.read_csv('../data/raw/bank.csv', sep=';')

## Nettoyage des données

### Petit Dataset

In [24]:
df_clean_small = df_small
df_clean_small = df_clean_small.rename(columns={"y":"subscribed", "default": "credit_in_default","contact":"contact_type", "pdays":"last_contact","previous":"number_of_contact_before_campaign","poutcome":"result_campaign" })

In [25]:
del df_clean_small["duration"]

In [26]:
df_clean_small = df_clean_small[df_clean_small['job'] != 'unknown']

In [27]:
# Création de la colonne contact_status

df_clean_small["contact_status"] = df_clean_small['last_contact'].apply(lambda x: 'no_contact' if x == -1 else 'already_contact')

In [28]:
df_clean_small.head()

,age,job,marital,education,credit_in_default,balance,housing,loan,contact_type,day,month,campaign,last_contact,number_of_contact_before_campaign,result_campaign,subscribed,contact_status
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,1,-1,0,unknown,no,no_contact
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,1,339,4,failure,no,already_contact
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,1,330,1,failure,no,already_contact
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,4,-1,0,unknown,no,no_contact
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact


### Grand Dataset

In [29]:
df_clean_large = df_large
df_clean_large = df_clean_large.rename(columns={"y":"subscribed", "default": "credit_in_default","contact":"contact_type", "pdays":"last_contact","previous":"number_of_contact_before_campaign","poutcome":"result_campaign" })

In [30]:
del df_clean_large["duration"]

In [31]:
df_clean_large = df_clean_large[df_clean_large['job'] != 'unknown']

In [32]:
df_clean_large["contact_status"] = df_clean_large['last_contact'].apply(lambda x: 'no_contact' if x == -1 else 'already_contact')

In [33]:
df_clean_large.head()

,age,job,marital,education,credit_in_default,balance,housing,loan,contact_type,day,month,campaign,last_contact,number_of_contact_before_campaign,result_campaign,subscribed,contact_status
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,-1,0,unknown,no,no_contact
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact
5,35,management,married,tertiary,no,231,yes,no,unknown,5,may,1,-1,0,unknown,no,no_contact


In [34]:
# Répartition des classes - Petit dataset
print("=== PETIT DATASET ===")
print("Nombre total d'observations:", len(df_clean_small))
print("\nRépartition brute:")
print(df_clean_small['subscribed'].value_counts())
print("\nRépartition en pourcentages:")
print(df_clean_small['subscribed'].value_counts(normalize=True) * 100)

print("\n" + "="*50)

# Répartition des classes - Grand dataset  
print("=== GRAND DATASET ===")
print("Nombre total d'observations:", len(df_clean_large))
print("\nRépartition brute:")
print(df_clean_large['subscribed'].value_counts())
print("\nRépartition en pourcentages:")
print(df_clean_large['subscribed'].value_counts(normalize=True) * 100)

=== PETIT DATASET ===
Nombre total d'observations: 4483

Répartition brute:
subscribed
no     3969
yes     514
Name: count, dtype: int64

Répartition en pourcentages:
subscribed
no     88.534464
yes    11.465536
Name: proportion, dtype: float64

=== GRAND DATASET ===
Nombre total d'observations: 44923

Répartition brute:
subscribed
no     39668
yes     5255
Name: count, dtype: int64

Répartition en pourcentages:
subscribed
no     88.302206
yes    11.697794
Name: proportion, dtype: float64


In [35]:
info_small = df_clean_small.info()
print(info_small)

<class 'pandas.core.frame.DataFrame'>
Index: 4483 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   age                                4483 non-null   int64 
 1   job                                4483 non-null   object
 2   marital                            4483 non-null   object
 3   education                          4483 non-null   object
 4   credit_in_default                  4483 non-null   object
 5   balance                            4483 non-null   int64 
 6   housing                            4483 non-null   object
 7   loan                               4483 non-null   object
 8   contact_type                       4483 non-null   object
 9   day                                4483 non-null   int64 
 10  month                              4483 non-null   object
 11  campaign                           4483 non-null   int64 
 12  last_contac

## Modélisation pour les critères de Campagne

### Selection des variables

In [36]:
variables_campagne = [
    'contact_type',
    'day',
    'month',
    'campaign',
    'last_contact',
    'number_of_contact_before_campaign',
    'result_campaign', 
    'subscribed',
    'contact_status']

# Création du dataset pour la modélisation (création d'une copie)
df_modelisation_small =  df_clean_small[variables_campagne].copy()

print("Dataset créé avec", len(variables_campagne)-1, "variables explicatives")
print("Taille du dataset:", df_modelisation_small.shape)

Dataset créé avec 8 variables explicatives
Taille du dataset: (4483, 9)


### Setup de Pycaret

In [37]:
# Environnement Pycaret

clf = setup(data=df_modelisation_small,
            target='subscribed',
            session_id=123,
           )


,Description,Value
0,Session id,123
1,Target,subscribed
2,Target type,Regression
3,Original data shape,"(4483, 9)"
4,Transformed data shape,"(4483, 25)"
5,Transformed train set shape,"(3138, 25)"
6,Transformed test set shape,"(1345, 25)"
7,Numeric features,4
8,Categorical features,4
9,Preprocess,True


## Analyse des résultats de configuration PyCaret

### Informations clés

* **Target :** `subscribed` (Binary : yes/no)
* **Mapping :** no=0, yes=1
* **Données originales :** 4,483 lignes, 9 colonnes
* **Données transformées :** 4,483 lignes, **25 colonnes**

### Transformation automatique des données

**Pourquoi 25 colonnes au lieu de 9 ?**

PyCaret a automatiquement appliqué un **one-hot encoding** des variables catégorielles :

* `contact_type` → contact_type_cellular, contact_type_telephone, etc.
* `month` → month_jan, month_feb, month_mar, etc.
* `result_campaign` → result_campaign_success, result_campaign_failure, etc.

### Division des données

* **Train (entraînement) :** 3,138 lignes (70%)
* **Test (évaluation) :** 1,345 lignes (30%)

Cette division permet d'entraîner les modèles sur 70% des données et de les évaluer sur les 30% restants pour mesurer leur performance sur des données non vues.

### Recherche du meilleur modèle

In [47]:
best_model = compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.8904,0.7514,0.8904,0.8652,0.8620,0.2152,0.2674,0.6230
rf,Random Forest Classifier,0.8862,0.6964,0.8862,0.8516,0.8528,0.1543,0.2018,0.3760
ada,Ada Boost Classifier,0.8862,0.7094,0.8862,0.8603,0.8643,0.2455,0.2763,0.2900
lightgbm,Light Gradient Boosting Machine,0.8853,0.7179,0.8853,0.8569,0.8597,0.2114,0.2493,0.3690
dummy,Dummy Classifier,0.8853,0.5000,0.8853,0.7837,0.8314,0.0000,0.0000,0.1490
et,Extra Trees Classifier,0.8681,0.6759,0.8681,0.8315,0.8441,0.1376,0.1527,0.3450
dt,Decision Tree Classifier,0.8152,0.5765,0.8152,0.8264,0.8202,0.1426,0.1439,0.1310
qda,Quadratic Discriminant Analysis,0.7763,0.6658,0.7763,0.8463,0.8000,0.2115,0.2253,0.1250
nb,Naive Bayes,0.7741,0.7145,0.7741,0.8466,0.8026,0.2099,0.2267,0.1380
lda,Linear Discriminant Analysis,0.7129,0.7119,0.7129,0.8489,0.7602,0.1794,0.2140,0.1410


# Comparaison GBC vs AdaBoost

## Gradient Boosting Classifier - Meilleur choix global

### Avantages de GBC sur AdaBoost
- **Accuracy supérieure** : 0.8913 vs 0.8875
- **AUC meilleure** : 0.7449 vs 0.7321 (séparation des classes optimisée)
- **Précision équilibrée** : 0.8677 vs 0.8603
- **Temps acceptable** : 0.658s vs 0.286s (performance justifie le coût)

### Résumé comparatif

| Métrique | GBC | AdaBoost | Avantage |
|----------|-----|----------|----------|
| Accuracy | 0.8913 | 0.8875 | GBC |
| AUC | **0.7449** | 0.7321 | **GBC** |
| Précision | 0.8677 | 0.8603 | GBC |
| Temps (s) | 0.658 | **0.286** | **AdaBoost** |

### Recommandation
**GBC** offre le meilleur équilibre performance/qualité pour gérer les données déséquilibrées avec une séparation des classes optimale.

# Comparaison GBC vs AdaBoost

## Gradient Boosting Classifier - Meilleur choix global

### Avantages de GBC sur AdaBoost
- **Accuracy supérieure** : 0.8913 vs 0.8875
- **AUC meilleure** : 0.7449 vs 0.7321 (séparation des classes optimisée)
- **Précision équilibrée** : 0.8677 vs 0.8603
- **Temps acceptable** : 0.658s vs 0.286s (performance justifie le coût)

### Résumé comparatif

| Métrique | GBC | AdaBoost | Avantage |
|----------|-----|----------|----------|
| Accuracy | 0.8913 | 0.8875 | GBC |
| AUC | **0.7449** | 0.7321 | **GBC** |
| Précision | 0.8677 | 0.8603 | GBC |
| Temps (s) | 0.658 | **0.286** | **AdaBoost** |

### Recommandation
**GBC** offre le meilleur équilibre performance/qualité pour gérer les données déséquilibrées avec une séparation des classes optimale.

In [40]:
from pycaret.classification import *
setup(data=df_clean_small, target='subscribed', fix_imbalance=True)  # Active SMOTE automatique

,Description,Value
0,Session id,8697
1,Target,subscribed
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(4483, 17)"
5,Transformed data shape,"(6901, 48)"
6,Transformed train set shape,"(5556, 48)"
7,Transformed test set shape,"(1345, 48)"
8,Numeric features,6
9,Categorical features,10


In [43]:
# 2. Optimiser les hyperparamètres
tuned_ada = tune_model(model_smote, optimize='AUC')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8758,0.7541,0.8758,0.8623,0.8679,0.3135,0.3175
1,0.8599,0.6334,0.8599,0.8339,0.8447,0.1691,0.1754
2,0.8790,0.7701,0.8790,0.8494,0.8578,0.2138,0.2348
3,0.8662,0.7554,0.8662,0.8237,0.8394,0.1024,0.1160
4,0.8758,0.6703,0.8758,0.8496,0.8584,0.2299,0.2441
5,0.9076,0.8013,0.9076,0.8953,0.8967,0.4443,0.4639
6,0.8981,0.6913,0.8981,0.8819,0.8850,0.3776,0.3973
7,0.8917,0.7144,0.8917,0.8751,0.8800,0.3580,0.3712
8,0.8786,0.7102,0.8786,0.8582,0.8654,0.2822,0.2926


Fitting 10 folds for each of 10 candidates, totalling 100 fits


### Rappel: La colonne Target => Déséquilibre

In [49]:
# Compter les occurrences de chaque classe
df_clean_small['subscribed'].value_counts(normalize=True) * 100

subscribed
no     88.534464
yes    11.465536
Name: proportion, dtype: float64

### Activation de SMOTE pour optimisation et Considération de F1 pour plus de pertinence

In [51]:
from pycaret.classification import *
setup(data=df_clean_small, target='subscribed', fix_imbalance=True)  # Active SMOTE automatique
best_model = compare_models(sort='F1')  # Trie par F1-Score plutôt qu'accuracy

,Description,Value
0,Session id,5144
1,Target,subscribed
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(4483, 17)"
5,Transformed data shape,"(6901, 48)"
6,Transformed train set shape,"(5556, 48)"
7,Transformed test set shape,"(1345, 48)"
8,Numeric features,6
9,Categorical features,10


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ada,Ada Boost Classifier,0.8875,0.7321,0.8875,0.8603,0.8653,0.2497,0.2788,0.2860
gbc,Gradient Boosting Classifier,0.8913,0.7449,0.8913,0.8677,0.8630,0.2199,0.2758,0.6580
lightgbm,Light Gradient Boosting Machine,0.8862,0.7258,0.8862,0.8564,0.8599,0.2103,0.2473,0.3480
rf,Random Forest Classifier,0.8843,0.7204,0.8843,0.8462,0.8501,0.1387,0.1827,0.3430
et,Extra Trees Classifier,0.8697,0.7012,0.8697,0.8372,0.8480,0.1647,0.1805,0.3270
dummy,Dummy Classifier,0.8853,0.5000,0.8853,0.7837,0.8314,0.0000,0.0000,0.1170
dt,Decision Tree Classifier,0.8219,0.5766,0.8219,0.8274,0.8245,0.1496,0.1500,0.1470
nb,Naive Bayes,0.7358,0.7081,0.7358,0.8449,0.7763,0.1818,0.2073,0.1220
lda,Linear Discriminant Analysis,0.7145,0.7133,0.7145,0.8464,0.7614,0.1740,0.2059,0.1400
ridge,Ridge Classifier,0.7119,0.7108,0.7119,0.8480,0.7598,0.1775,0.2113,0.1190


# Analyse AdaBoost - Choix du modèle

## Pourquoi AdaBoost reste le meilleur choix pour notre dataset

### Performance déjà excellente
- **F1-score de 0.865** sur les tests initiaux
- **Recall de 0.888** - crucial pour notre déséquilibre de classes (88.5% vs 11.5%)
- **Temps d'entraînement rapide** (0.286s) - avec 44k entrées, cela restera gérable

## Conclusion
AdaBoost offre le meilleur F1-score (métrique cruciale pour les données déséquilibrées) avec une vitesse d'entraînement optimale. Avec 44k entrées, nous pouvons l'optimiser davantage pour atteindre nos objectifs de performance.

,Description,Value
0,Session id,1195
1,Target,subscribed
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(4483, 17)"
5,Transformed data shape,"(6901, 48)"
6,Transformed train set shape,"(5556, 48)"
7,Transformed test set shape,"(1345, 48)"
8,Numeric features,6
9,Categorical features,10


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8662,0.6970,0.8662,0.8237,0.8394,0.1024,0.1160
1,0.8981,0.7179,0.8981,0.8811,0.8716,0.2679,0.3332
2,0.8885,0.7419,0.8885,0.8586,0.8537,0.1545,0.2165
3,0.8790,0.7569,0.8790,0.8494,0.8578,0.2138,0.2348
4,0.8758,0.7098,0.8758,0.8461,0.8556,0.2057,0.2231
5,0.8790,0.7205,0.8790,0.8422,0.8512,0.1602,0.1890
6,0.8758,0.6728,0.8758,0.8461,0.8556,0.2057,0.2231
7,0.8885,0.7009,0.8885,0.8699,0.8753,0.3293,0.3438
8,0.9010,0.7427,0.9010,0.8850,0.8797,0.3262,0.3767


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8822,0.6852,0.8822,0.8500,0.8569,0.1960,0.2263
1,0.8854,0.7410,0.8854,0.8571,0.8623,0.2306,0.2613
2,0.8949,0.7331,0.8949,0.8747,0.8658,0.2314,0.2976
3,0.8662,0.7848,0.8662,0.8381,0.8490,0.1831,0.1927
4,0.8822,0.7163,0.8822,0.8500,0.8569,0.1960,0.2263
5,0.8854,0.7467,0.8854,0.8524,0.8555,0.1763,0.2194
6,0.8790,0.6862,0.8790,0.8526,0.8607,0.2381,0.2553
7,0.8758,0.7016,0.8758,0.8461,0.8556,0.2057,0.2231
8,0.9010,0.7754,0.9010,0.8866,0.8768,0.3029,0.3663


Fitting 10 folds for each of 9 candidates, totalling 90 fits
AdaBoostClassifier(algorithm='SAMME.R', estimator=None, learning_rate=0.8,
                   n_estimators=50, random_state=1195)
